# OpenVLA transfer arm — Colab T4 smoke test

Proves the **robot-pretrained OpenVLA-7B transfer arm** loads in 4-bit and trains a few
steps on a **free Colab T4** without OOM/dtype errors. This is the cheap way to burn down
the *pending lab-GPU validation* blocker before booking the real GPU.

**Scope / what this is NOT.** This is a ~50-step smoke run in `velocity` mode — not a
result. The `scratch` and `prismatic` control arms need >=24 GB bf16 and will **not** fit
a T4; run the real 3-arm ablation on the lab GPU with `configs/openvla.yaml`.

### Before you start
1. **Runtime -> Change runtime type -> T4 GPU.**
2. **Push the repo first.** This notebook clones `main` from GitHub, so the fp16 changes
   (`configs/openvla_colab.yaml`, the `precision`/`max_steps` support) must already be pushed.
3. **Upload the data to Google Drive.** The 572 MB `data/uzh_fpv/` is git-ignored. On your
   laptop, from the repo root:
   ```bash
   tar czf uzh_fpv.tar.gz -C data uzh_fpv
   ```
   then upload `uzh_fpv.tar.gz` to Drive at `MyDrive/alpamayo/uzh_fpv.tar.gz` (adjust the
   path in the data cell if you put it elsewhere).

## 1. Confirm the GPU is a T4
Turing (T4) / Pascal (P100) have no bf16 tensor cores — that's why this run uses fp16.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Clone the repo

In [ ]:
%cd /content
!rm -rf alpamayo_drone
!git clone https://github.com/AnuragMandke/alpamayo_drone.git
%cd /content/alpamayo_drone
!git log --oneline -1

## 3. Install the OpenVLA env deps
OpenVLA's `trust_remote_code` needs **transformers 4.40.1** (Colab ships a much newer one, so
this downgrades it). `accelerate` and `peft` are **pinned to that same era, not floored** —
Colab preinstalls versions that satisfy a `>=` floor, and a modern accelerate moves 4-bit
models with `.to()`, which transformers 4.40.1 forbids (it crashes *after* the 15 GB load).

We deliberately **do not** reinstall torch/torchvision — Colab's preinstalled build is matched
to the runtime CUDA, and replacing it from PyPI often breaks the GPU.

The version print below is the check that the downgrade actually took effect. **If it shows
anything else, restart the session** — pip cannot replace an already-imported module.

In [ ]:
!pip install -q \
  "transformers==4.40.1" "tokenizers>=0.19,<0.20" "timm==0.9.10" \
  "accelerate==0.29.3" "peft==0.11.1" "bitsandbytes>=0.43.0" \
  "Pillow>=10.0.0" "PyYAML>=6.0"

import transformers, accelerate, peft, bitsandbytes
print(f'transformers={transformers.__version__} accelerate={accelerate.__version__} '
      f'peft={peft.__version__} bitsandbytes={bitsandbytes.__version__}')
print('Expect transformers=4.40.1 accelerate=0.29.3 peft=0.11.1. If not, the old versions '
      'are still imported: Runtime -> Restart session, then re-run from cell 2.')

## 4. Bring the data over from Drive
Mounts Drive and extracts the tarball you uploaded into `data/uzh_fpv/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

TARBALL = '/content/drive/MyDrive/alpamayo/uzh_fpv.tar.gz'  # <-- edit if you used another path
!mkdir -p data
!tar xzf "$TARBALL" -C data
!echo 'trajectories:' && ls data/uzh_fpv/trajectories | wc -l && ls data/uzh_fpv/trajectories | head -3

## 5. Point the HF cache at local disk
OpenVLA weights (~15 GB) download here. This is on Colab's fast ephemeral disk and is
**re-downloaded every session** — a free 15 GB Drive can't hold it, so we don't persist it.

In [ ]:
import os
os.environ['HF_HOME'] = '/content/hf_cache'

## 6. Run the smoke test
First run spends several minutes downloading the ~15 GB of weights, then trains 50 steps.
The `pretrained` arm is the only one that fits a 16 GB T4 (4-bit QLoRA).

In [ ]:
!python scripts/train_openvla.py --config configs/openvla_colab.yaml --init pretrained

## What success looks like
- `[Train] precision=torch.float16 (autocast=on)` printed near the top.
- Trainable-params line shows only the LoRA adapters (a few tens of millions, <1%).
- A finite, non-NaN `loss=` that drops sharply over the first ~15 steps and then
  **plateaus** around 2–3 nats. This log is a *windowed* mean (mean of the last N
  batches), so the plateau is the honest picture — 50 steps x batch 4 is 200 samples,
  under 2% of one epoch, so a plateau is expected and is **not** failure.
- Ends with `[max_steps=50 reached — stopping]` and saves
  `outputs/openvla/pretrained/epoch001` (~130 MB — adapters only).

## Troubleshooting
- **CUDA OOM** — lower `batch_size` to 2 (or 1) in `configs/openvla_colab.yaml`.
- **`loss=nan`** — fp16 underflow; the GradScaler should handle it, but if it persists
  drop `optimizer.lr` and/or `batch_size`.
- **`` `.to` is not supported for `4-bit` ``, at the end of `from_pretrained`** — your
  `accelerate` is newer than `transformers==4.40.1`. Re-run the install cell (it pins
  `accelerate==0.29.3`) and **restart the session**; check the version print.
- **bf16 error inside the OpenVLA remote code** — the remote `modeling_prismatic.py` may
  hardcode a bf16 path in the vision tower. If so, that's exactly the finding this smoke
  test exists to surface — report it; the fix is a small patch to force the vision dtype.
- **Session disconnects mid-download** — free Colab idles out; keep the tab active, or the
  `max_steps` cap keeps the whole run short enough to finish in one session.